# ASL Fingerspelling — Transformer (BERT-like) 

**What this notebook does**

- Loads preprocessed landmark `.npy` sequences from a local Kaggle input path (adjustable)
- Builds a small Transformer encoder (BERT-like) as a sequence model (no external HF download)
- Trains with `CTCLoss` (correct `input_lengths` handling)
- Uses PyTorch Lightning for training loop, checkpointing and LR logging

**How to use**

1. Upload this notebook to Kaggle (or run locally where dataset is available).
2. Set `KAGGLE_BASE` path if your landmarks are in a different folder.
3. Run all cells. The notebook is intentionally small so it runs quickly for debugging.

---


In [29]:
# Cell 1 — Setup: install or confirm packages (Kaggle usually has these preinstalled)
# If running on a clean environment you may need to install pytorch-lightning.
# Uncomment following line on environments without PL preinstalled:
# !pip install -q pytorch-lightning

print("torch:", __import__('torch').__version__)
print("pytorch_lightning:", __import__('pytorch_lightning').__version__)


torch: 2.6.0+cu124
pytorch_lightning: 2.5.2


In [30]:
import os
import warnings

# Silence CUDA/XLA warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress XLA/TensorFlow logs
warnings.filterwarnings("ignore")          # ignore Python warnings

In [31]:
# Cell 2 — Imports and config
import os
import math
import glob
import numpy as np
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from sklearn.metrics import accuracy_score
import torch

# Simple config — tweak as needed
KAGGLE_BASE = "/kaggle/input"  # change if your data is somewhere else (e.g., /mnt/data/)
LANDMARK_GLOB = os.path.join(KAGGLE_BASE, "asl-fingerspelling-preprocessing-train-dataset", "train_landmarks_npy", "*", "*.npy")
BATCH_SIZE = 8
EPOCHS = 3
LR = 3e-4
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Looking for landmarks with glob:', LANDMARK_GLOB)


Device: gpu
Looking for landmarks with glob: /kaggle/input/asl-fingerspelling-preprocessing-train-dataset/train_landmarks_npy/*/*.npy


In [32]:
# Cell 3 — Small dataset loader for .npy landmark sequences
# Assumes each sample file is a numpy array: [T, feat_dim]
# And that labels are derivable from the directory name (or from a parallel labels file).
# We'll create a toy label mapping for the purposes of this notebook — replace with actual mapping if available.

class LandmarksDataset(Dataset):
    def __init__(self, file_paths: List[str], label_map: dict, max_len: int = 100):
        self.files = file_paths
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        arr = np.load(path)  # shape: [T, feat_dim]
        T, feat = arr.shape
        # truncate or pad to max_len
        if T > self.max_len:
            arr = arr[:self.max_len]
            T = self.max_len
        else:
            pad = np.zeros((self.max_len - T, feat), dtype=arr.dtype)
            arr = np.concatenate([arr, pad], axis=0)

        # Fake label: use parent folder name as label (replace with proper label loader)
        # e.g., /.../A/123.npy -> letter 'A'
        parent = os.path.basename(os.path.dirname(path))
        # if not in label_map, assign 0
        label_token = self.label_map.get(parent, 0)
        # create a simple target sequence (for demo): repeated token (you should replace this)
        target = np.array([label_token], dtype=np.int64)

        sample = {
            "x": torch.tensor(arr, dtype=torch.float32),      # [T, feat]
            "x_len": torch.tensor(min(T, self.max_len), dtype=torch.long),
            "y": torch.tensor(target, dtype=torch.long),     # 1D target
            "y_len": torch.tensor(len(target), dtype=torch.long)
        }
        return sample

# collect files (if none found, create synthetic data)
files = sorted(glob.glob(LANDMARK_GLOB))[:500]  # limit to 500 for speed
if len(files) == 0:
    print("No landmark files found at the glob path. Creating synthetic files for demo.")
    os.makedirs('/kaggle/working/synthetic_landmarks', exist_ok=True)
    files = []
    for i in range(200):
        T = np.random.randint(20, 80)
        feat = 84
        arr = np.random.randn(T, feat).astype(np.float32)
        path = f"/kaggle/working/synthetic_landmarks/sample_{i}.npy"
        np.save(path, arr)
        files.append(path)

# create a toy label map (replace with actual mapping)
unique_parents = sorted({os.path.basename(os.path.dirname(f)) for f in files})
label_map = {p: i+1 for i, p in enumerate(unique_parents)}  # reserve 0 for blank in CTC

# split
split = int(0.8 * len(files))
train_files, val_files = files[:split], files[split:]
print("Train files:", len(train_files), "Val files:", len(val_files))


Train files: 400 Val files: 100


In [33]:
# Cell 4 — Collate fn & DataLoaders
def collate_fn(batch):
    # batch is list of samples with keys x, x_len, y, y_len
    xs = torch.stack([s['x'] for s in batch], dim=0)   # [B, T, feat]
    x_lens = torch.stack([s['x_len'] for s in batch]).to(torch.long)  # [B]
    ys = [s['y'] for s in batch]
    y_lens = torch.stack([s['y_len'] for s in batch]).to(torch.long)

    # flatten targets into 1D as required by torch.ctc_loss
    targets = torch.cat(ys).to(torch.long)

    return {"x": xs, "x_len": x_lens, "y": targets, "y_len": y_lens}

train_ds = LandmarksDataset(train_files, label_map, max_len=100)
val_ds = LandmarksDataset(val_files, label_map, max_len=100)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("One batch shapes (for quick sanity):")
batch = next(iter(train_loader))
print({k: v.shape for k, v in batch.items()})


One batch shapes (for quick sanity):
{'x': torch.Size([8, 100, 390]), 'x_len': torch.Size([8]), 'y': torch.Size([8]), 'y_len': torch.Size([8])}


In [34]:
# Cell 5 — Small Transformer encoder (BERT-like) and LightningModule with correct CTC usage
class SmallTransformerEncoder(nn.Module):
    def __init__(self, input_dim, d_model=256, nhead=8, num_layers=3, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, len(label_map) + 1)  # +1 for CTC blank (id=0)

    def forward(self, x, src_key_padding_mask=None):
        # x: [B, T, feat]
        x = self.input_proj(x)
        # Transformer expects [B, T, D] with batch_first=True (we used batch_first)
        out = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        logits = self.output_proj(out)  # [B, T, C]
        return logits

class LitCTC(pl.LightningModule):
    def __init__(self, model, lr=LR):
        super().__init__()
        self.model = model
        self.ctc = nn.CTCLoss(blank=0, zero_infinity=True)
        self.lr = lr

    def forward(self, x, mask=None):
        return self.model(x, src_key_padding_mask=mask)

    @staticmethod
    def make_padding_mask(x_len, max_T):
        # x_len: [B] lengths (not padded)
        # returns mask shaped [B, T] where True indicates padding positions
        rng = torch.arange(max_T, device=x_len.device)[None, :].expand(len(x_len), max_T)
        return rng >= x_len[:, None]

    def training_step(self, batch, batch_idx):
        x = batch['x'].to(self.device)         # [B, T, feat]
        x_len = batch['x_len'].to(self.device) # [B]
        y = batch['y'].to(self.device)         # concatenated targets
        y_len = batch['y_len'].to(self.device) # [B]

        B, T, _ = x.shape
        pad_mask = self.make_padding_mask(x_len, T)  # [B, T]

        logits = self.model(x, src_key_padding_mask=pad_mask)  # [B, T, C]
        logp = logits.log_softmax(-1)                         # [B, T, C]

        # input_lengths must be shape (B,)
        input_lengths = x_len.view(-1).to(torch.long)
        if input_lengths.numel() != B:
            input_lengths = torch.full((B,), T, dtype=torch.long, device=self.device)

        loss = self.ctc(logp.permute(1,0,2), y, input_lengths, y_len)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch['x'].to(self.device)
        x_len = batch['x_len'].to(self.device)
        y = batch['y'].to(self.device)
        y_len = batch['y_len'].to(self.device)

        B, T, _ = x.shape
        pad_mask = self.make_padding_mask(x_len, T)

        logits = self.model(x, src_key_padding_mask=pad_mask)
        logp = logits.log_softmax(-1)

        input_lengths = x_len.view(-1).to(torch.long)
        if input_lengths.numel() != B:
            input_lengths = torch.full((B,), T, dtype=torch.long, device=self.device)

        loss = self.ctc(logp.permute(1,0,2), y, input_lengths, y_len)
        self.log('val_loss', loss, prog_bar=True)

        # quick greedy decode for CER approx (optional, simple)
        with torch.no_grad():
            pred = logp.argmax(-1).cpu().numpy()  # [B, T]
            # collapse repeats and remove blanks (0)
            preds = []
            for b in range(B):
                seq = []
                last = -1
                for t in range(T):
                    token = int(pred[b, t])
                    if token != last and token != 0:
                        seq.append(str(token))
                    last = token
                preds.append(''.join(seq))
            # build gts from concatenated y by splitting using y_len
            gts = []
            idx = 0
            for l in y_len.cpu().numpy():
                gts.append(''.join(str(int(x)) for x in y.cpu().numpy()[idx:idx+l]))
                idx += l
            # very simple CER by chr. Levenshtein (not optimized)
            def cer(a, b):
                m, n = len(a), len(b)
                dp = [[0]*(n+1) for _ in range(m+1)]
                for i in range(m+1):
                    dp[i][0] = i
                for j in range(n+1):
                    dp[0][j] = j
                for i in range(1, m+1):
                    for j in range(1, n+1):
                        cost = 0 if a[i-1]==b[j-1] else 1
                        dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
                return dp[m][n] / max(1, n)
            cer_val = sum(cer(p, g) for p, g in zip(preds, gts)) / len(gts)
            self.log('val_cer', cer_val, prog_bar=True)

        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        return opt

# instantiate model
sample_feat = next(iter(train_loader))['x'].shape[-1]
model = SmallTransformerEncoder(input_dim=sample_feat)
lit = LitCTC(model)
print("Model ready. Params:", sum(p.numel() for p in lit.parameters()))

Model ready. Params: 1681922


In [35]:
# Cell 6 — Trainer and callbacks (Kaggle-ready)
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
import torch
import pytorch_lightning as pl
import os
import warnings

# Silence warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress XLA/TensorFlow logs
warnings.filterwarnings("ignore")          # ignore Python warnings

# Callbacks
checkpoint_cb = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=True,  # ensures a checkpoint is saved even if val_loss didn't improve
    filename='bertctc-{epoch:02d}-{val_loss:.4f}'
)
lr_cb = LearningRateMonitor(logging_interval='step')

# Detect device
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'

# Trainer
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=[checkpoint_cb, lr_cb],
    accelerator=DEVICE,
    devices=1,  # single device to avoid errors
    precision='16-mixed' if torch.cuda.is_available() else 32,
    log_every_n_steps=10,
)

# Sanity prints
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

# Run training
trainer.fit(lit, train_loader, val_loader)

print("Training complete ✅ Checkpoint saved (last epoch)")


Train batches: 50
Val batches: 13


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Training complete ✅ Checkpoint saved (last epoch)


In [36]:
# Cell 7 — Load latest checkpoint or use current trained model
from pathlib import Path

# Try to load checkpoint
ckpt_files = list(Path(".").glob("bertctc-*.ckpt"))

if ckpt_files:
    latest = sorted(ckpt_files)[-1]
    print("Found checkpoint:", latest)
    lit2 = LitCTC.load_from_checkpoint(
        str(latest),
        model=SmallTransformerEncoder(input_dim=sample_feat)
    )
else:
    print("No checkpoint found — using current trained model")
    lit2 = lit  # fallback: use current session model

lit2.eval()
lit2.freeze()
print("Model ready for evaluation ✅")


No checkpoint found — using current trained model
Model ready for evaluation ✅


In [37]:
# Cell 8 — Robust validation accuracy evaluation

all_preds = []
all_labels = []

for batch in val_loader:
    x = batch['x'].to(lit2.device)
    y = batch['y'].cpu().numpy()
    
    with torch.no_grad():
        logits = lit2(x)  # [B, T, C]
        pred = torch.argmax(logits, dim=-1)  # [B, T]
        
        # Process each sequence in batch
        batch_preds = []
        for seq in pred:
            # Remove blanks (0) and take first non-blank token (or 0 if all blank)
            non_blanks = seq[seq != 0]
            if len(non_blanks) > 0:
                batch_preds.append(non_blanks[0].item())  # take first non-blank
            else:
                batch_preds.append(0)  # fallback to blank if all predictions are blank
    
    all_preds.extend(batch_preds)
    all_labels.extend(y)

# Accuracy
acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc*100:.2f}%")

Validation Accuracy: 0.00%
